# Text-to-Speech: 텍스트를 음성으로 변환하기

TTS(Text-to-Speech)는 입력 텍스트를 사람이 들을 수 있는 음성으로 변환하는 기능이다. 안내 방송, 접근성 읽기, 콘텐츠 내레이션에 사용할 수 있다.

이 노트북에서는 텍스트, 음성 모델, 목소리를 선택해 MP3 파일을 만든다. 생성한 `output.mp3`는 다음 STT 수업의 입력 파일로 사용한다.

## 모델과 목소리 선택

- 단순 파일 TTS는 `tts-1` 또는 `tts-1-hd`를 사용한다. `tts-1`은 낮은 지연 시간, `tts-1-hd`는 음성 품질 비교에 사용한다.
- 지시형 TTS는 `gpt-4o-mini-tts`를 사용해 톤·억양·속도 같은 음성 지시를 추가할 수 있다. 공식 TTS 가이드는 최신 지시형 모델로 소개하지만 모델 카탈로그는 Deprecated로 표시하므로, 실행 전 두 문서의 상태를 함께 확인한다.
- audio in/out이 필요한 Chat Completions 요청은 `gpt-audio-1.5`를 사용한다. Speech API의 단순 파일 생성과는 별도 경로이다.
- 실시간 음성 에이전트는 Realtime API와 `gpt-realtime-2.1`을 사용한다. 이 노트북의 파일 생성·재생 흐름과 구분한다.
- `voice`는 말투와 음색을 선택한다. 이 예제는 `nova`를 사용한다.

### 모델별 비용

- `tts-1`: 입력 문자 100만 자당 15달러이다. 단순 비례 계산으로 1,000자는 약 0.015달러이다.
- `tts-1-hd`: 입력 문자 100만 자당 30달러이다. 단순 비례 계산으로 1,000자는 약 0.03달러이다.
- 짧은 문장 한두 개의 비용은 작지만, 여러 학생이 반복 실행하면 전체 입력 글자 수에 따라 비용이 누적된다.
- 가격은 변경될 수 있으므로 [TTS-1 모델 가격](https://developers.openai.com/api/docs/models/tts-1)과 [TTS-1 HD 모델 가격](https://developers.openai.com/api/docs/models/tts-1-hd)을 실행 전 확인한다.

생성 음성이 AI로 만들어졌다는 사실을 사용자에게 명확히 알린다. 한국어는 발음, 억양, 고유명사, 문장 끝의 자연스러움을 함께 확인한다.

공식 문서는 다음과 같다.

- [Text-to-Speech 가이드](https://developers.openai.com/api/docs/guides/text-to-speech)
- [Speech 생성 API](https://developers.openai.com/api/reference/resources/audio/subresources/speech/methods/create)
- [모델 카탈로그](https://developers.openai.com/api/docs/models)
- [GPT-4o mini TTS 모델](https://developers.openai.com/api/docs/models/gpt-4o-mini-tts)
- [GPT audio 1.5 모델](https://developers.openai.com/api/docs/models/gpt-audio-1.5)
- [GPT Realtime 2.1 모델](https://developers.openai.com/api/docs/models/gpt-realtime-2.1)


### Speech API 요청의 입력과 출력

음성 생성 요청은 `model`, `voice`, `input`을 함께 사용한다. 텍스트 입력은 음성 바이트로 변환되고 MP3 파일로 저장되어 재생 또는 파일 전사의 입력이 된다.


### API 클라이언트 준비

이미 설정한 `.env`를 불러온다. 키 값은 출력하지 않는다.

- `find_dotenv(usecwd=True)`: 현재 작업 폴더부터 `.env` 경로를 찾는다.
- `load_dotenv(dotenv_path, override=False)`: 찾은 값을 불러오되 이미 있는 환경 변수는 유지한다.


In [1]:
import os

from dotenv import find_dotenv, load_dotenv

dotenv_path = find_dotenv(usecwd=True)
if not dotenv_path:
    raise FileNotFoundError("08_llm 프로젝트 최상위의 .env 파일을 확인한다.")
load_dotenv(dotenv_path, override=False)

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError(".env의 OPENAI_API_KEY를 확인한다.")

print("환경 변수 준비 완료")


환경 변수 준비 완료


### 텍스트를 MP3 파일로 저장하기

`input`의 한국어 문장을 Speech API에 전달하면 선택한 모델과 목소리가 음성 데이터를 만든다. 스트리밍 응답은 `output.mp3`에 저장되고, 다음 셀의 `Audio`와 다음 STT 수업의 전사 입력으로 사용한다.

[모델 카탈로그](https://developers.openai.com/api/docs/models)

- `model`: 음성 생성 모델로 `tts-1`을 전달한다.
- `voice`: `nova` 음색을 선택한다.
- `input`: 음성으로 바꿀 한국어 텍스트이다.
- `with_streaming_response.create(...)`: 응답 음성 바이트를 스트리밍으로 받는다. `response_format`을 생략하면 기본 MP3 형식을 사용한다.
- `stream_to_file(file_path)`: 스트리밍 응답을 `output.mp3`에 저장한다.


In [2]:
from openai import OpenAI
client = OpenAI()

# 출력 경로
file_path = "output.mp3"

with client.audio.speech.with_streaming_response.create(   #with 구문: close를 알아서 해주는.
    model='tts-1',
    voice='nova',
    input='점심을 먹으러 나가야 하는데 밖이 너무 덥다. 뭐 먹지?'
) as response:
    response.stream_to_file(file_path)

### 생성한 MP3 재생하기

`Audio`는 앞 셀에서 저장한 `output.mp3`를 읽어 노트북에 재생 컨트롤을 표시한다. 재생 결과로 한국어 발음과 억양을 확인하고, 같은 파일을 STT 입력으로 넘긴다.

- `Audio(data, autoplay=False)`: 첫 번째 위치 인자 `data`에 파일 경로 또는 오디오 데이터를 받아 재생 객체를 만든다. 이 셀은 `file_path`를 전달한다.
- `autoplay=True`: 재생 컨트롤이 표시될 때 자동 재생을 요청한다. 브라우저 정책에 따라 사용자의 재생 동작이 필요할 수 있다.


In [3]:
from IPython.display import Audio
Audio(file_path, autoplay=True)

# 속도를 조절 가능한 STT API 함수 만들기

In [4]:
from pathlib import Path

def create_speech(text, output_path, model='tts-1', voice='nova', speed=1.0):

    # text가 문자열이 아니거나, 문장이 비어있는 경우
    if not isinstance(text, str) or not text.strip():
        raise ValueError('text는 공백이 아닌 문장을 전달해주세요')

    # 0.25~4.0 범위가 아니라면
    if not 0.25 <= speed <= 4.0:
        raise ValueError('speed는 2.5~4.0 범위로 지정해주세요.')

    # output_path 형태가 절대경로인 경우
    # -> 우리는 상대경로로만 작성하여 현재 노트북 파일과 같은 위치에 생성 예정
    output_path = Path(output_path)
    if output_path.is_absolute():
        raise ValueError("output_path는 상대 경로 형태로만 작성해주세요.")

    # Speech API 호출하여 MP3 스트리밍 응답 받기
    with client.audio.speech.with_streaming_response.create(
        model=model,
        voice=voice,
        input=text,
        response_format='mp3',
        speed=speed
    ) as response:
        response.stream_to_file(output_path)

    return output_path


In [5]:
text = """
Like me, Like me
아주 눈이 부신
너를 숨김없이 보여줘

한 번도 빛난 적 없었던
미지의 향으로
온 세상을 물들여

새로워진 장면에
두 눈앞은 황홀해
너의 손을 잡을 땐
너와 어우러질 땐

빛을 이끌어 With me, With me
마치 Chemically 우린
완벽하게 어울려

Feeling love attack!
"""

slow_path = create_speech(text, "slow.mp3", speed=0.7)
normal_path = create_speech(text, "normal.mp3", speed=1.0)
fast_path = create_speech(text, "fast.mp3", speed=2.0)

display(Audio(slow_path, autoplay=False))
display(Audio(normal_path, autoplay=False))
display(Audio(fast_path, autoplay=False))